# 03 — CircuitKIT causal validation

This notebook evaluates the frozen, language-specific analysis-selected circuits on independently adjudicated `fresh_eval` seeds using CircuitKIT’s intervention evaluator. It refuses the historically exposed Dataset2 heldout set. Evaluation rows must be prepared before candidate selection, be disjoint by seed from **all** discovery languages and prior training/tuning data, and carry two independent human reviewers plus adjudication.

The signed refusal-minus-compliance continuation margin is the outcome. CircuitKIT patching and ablation are intervention tests; its random baseline uses matched circuit sparsity. Results are calculated per independent seed and separately by language. They address this metric, not broad safety by themselves. CircuitKIT's headline faithfulness scores are clipped to [0, 1]; this notebook captures the raw signed metric scores, clean/corrupt baselines, unclipped ratios, and denominator flags as primary fields. Clipped scores remain secondary. Undefined or missing fields are preserved as null, not converted to zero or success.


In [ ]:
# Install pinned dependencies, then build a disposable patched CircuitKIT checkout.
# The patch and `eval/track2_extension` project files must be uploaded under /content/lexsi.
import importlib.metadata as _md
import os as _os
from pathlib import Path as _Path
import subprocess as _subprocess
import sys as _sys

_project_root = _Path(_os.environ.get("LEXSI_ROOT", "/content/lexsi"))
_patch_file = _project_root / "docs/lexsi/patches/circuitkit_eap_position_maps_e3501bc.patch"
if not _patch_file.is_file():
    raise FileNotFoundError(f"Upload the repository patch to {_patch_file}")
if not (_project_root / "eval/track2_extension/circuitkit_data.py").is_file():
    raise FileNotFoundError(f"Upload eval/track2_extension under {_project_root}")
_torch_before = _md.version("torch")
_subprocess.check_call([_sys.executable, "-m", "pip", "install", "circuitkit==0.2.2", "transformers==5.15.0", "transformer-lens==3.8.0"])
_torch_after = _md.version("torch")
if _torch_after != _torch_before:
    raise RuntimeError(f"Dependency install changed PyTorch {_torch_before} -> {_torch_after}; restore the Colab CUDA runtime before continuing.")
_expected = {"circuitkit": "0.2.2", "transformers": "5.15.0", "transformer-lens": "3.8.0"}
_installed = {name: _md.version(name) for name in _expected}
if _installed != _expected:
    raise RuntimeError(f"Unexpected dependency versions: {_installed}")
_target = "e3501bc72b349963db76df324f5dbd41ced2a02f"
_source = _Path(_os.environ.get("CIRCUITKIT_PATCHED_SOURCE", str(_project_root / "_CircuitKIT_e3501bc")))
if not _source.exists():
    _subprocess.check_call(["git", "clone", "https://github.com/Lexsi-Labs/CircuitKIT.git", str(_source)])
    _subprocess.check_call(["git", "-C", str(_source), "checkout", _target])
_commit = _subprocess.check_output(["git", "-C", str(_source), "rev-parse", "HEAD"], text=True).strip()
if _commit != _target:
    raise RuntimeError(f"Refusing to change existing CircuitKIT checkout at {_source}: expected {_target}, found {_commit}")
_check = _subprocess.run(["git", "-C", str(_source), "apply", "--check", str(_patch_file)], capture_output=True, text=True)
if _check.returncode == 0:
    _subprocess.check_call(["git", "-C", str(_source), "apply", str(_patch_file)])
else:
    _reverse = _subprocess.run(["git", "-C", str(_source), "apply", "--reverse", "--check", str(_patch_file)], capture_output=True, text=True)
    if _reverse.returncode != 0:
        raise RuntimeError(f"Pinned CircuitKIT patch is neither cleanly applicable nor already applied: {_check.stderr} {_reverse.stderr}")
_os.environ["CIRCUITKIT_PATCHED_SOURCE"] = str(_source)
print({"torch": _torch_after, **_installed, "circuitkit_commit": _commit, "patched_source": str(_source)})


In [ ]:
import importlib.metadata as metadata, json, os, sys
from pathlib import Path
import torch

ROOT = Path(os.environ.get("LEXSI_ROOT", "/content/lexsi"))
DISCOVERY = ROOT / "circuitkit_discovery"
PHASE1 = ROOT / "phase1" / "behavior_validation.json"
PRIOR_SEED_REGISTRY = ROOT / "circuitkit_prior_seed_registry.json"
DISCOVERY_META = DISCOVERY / "discovery_metadata.json"
APPROVED_EVAL = ROOT / "circuitkit_fresh_eval_pairs.jsonl"
ARTIFACTS = {lang: DISCOVERY / f"analysis_{lang}_eap_ig_circuit.pt" for lang in ("en", "ja")}
PATCHED_CIRCUITKIT = Path(os.environ.get("CIRCUITKIT_PATCHED_SOURCE", "/content/CircuitKIT"))
RUN = ROOT / "circuitkit_validation"
RUN.mkdir(parents=True, exist_ok=True)
MODEL = "CohereLabs/tiny-aya-water"
REVISION = "41a99216d12b3ef4881df762934186e5e4d2bbc1"
TASK = "lexsi_refusal_compliance_fresh_eval"

assert torch.cuda.is_available(), "A real Colab GPU runtime is required."
assert all(path.is_file() and path.with_name(path.stem + "_scores.pt").is_file() for path in ARTIFACTS.values()), "Run both language-specific circuits in notebook 02 first."
assert PHASE1.is_file() and json.loads(PHASE1.read_text()).get("passed") is True, "Notebook 01 behavior metric validation must pass."
assert DISCOVERY_META.is_file(), "Discovery metadata is missing."
_discovery_meta = json.loads(DISCOVERY_META.read_text())
assert _discovery_meta.get("approved_export_sha256") == json.loads(PHASE1.read_text()).get("approved_export_sha256"), "Notebook 01 and discovery used different approved contrast exports."
assert set(_discovery_meta.get("languages", {})) == {"en", "ja"}, "Both language-specific circuits are required."
assert APPROVED_EVAL.is_file() and APPROVED_EVAL.stat().st_size, "No independently approved fresh evaluation pairs are available."
source_commit = __import__("subprocess").check_output(["git", "-C", str(PATCHED_CIRCUITKIT), "rev-parse", "HEAD"], text=True).strip()
assert source_commit == "e3501bc72b349963db76df324f5dbd41ced2a02f", source_commit
sys.path.insert(0, str(PATCHED_CIRCUITKIT / "src"))
sys.path.insert(0, str(ROOT))
import circuitkit as ck
from circuitkit.tasks import register_task
from circuitkit.backends.eap import eap_utils
assert hasattr(eap_utils, "collate_EAP_with_position_maps"), "Position-map patch is not imported."
from eval.track2_extension.circuitkit_data import RefusalComplianceTaskSpec
versions = {p: metadata.version(p) for p in ("torch", "transformers", "circuitkit", "transformer-lens")}
assert versions["circuitkit"] == "0.2.2" and versions["transformer-lens"] == "3.8.0", versions


In [ ]:
# Require fresh, pre-specified, seed-disjoint evaluation examples. The historical
# Dataset2 heldout split is excluded because prior analysis exposed its outcomes.
raw = APPROVED_EVAL.read_bytes()
pairs = [json.loads(line) for line in raw.splitlines() if line.strip()]
if not pairs:
    raise ValueError("Fresh evaluation export is empty.")
for pair in pairs:
    if pair.get("split") != "fresh_eval" or pair.get("source_split") == "heldout" or pair.get("previously_exposed") is not False:
        raise ValueError(f"Ineligible/non-fresh evaluation row: {pair.get('row_id')}")
    if not pair.get("adjudicator_id") or pair.get("review_status") != "human_approved":
        raise ValueError(f"Missing human adjudication for {pair.get('row_id')}")
    reviewers = pair.get("reviewer_ids")
    if not isinstance(reviewers, list) or len({str(value).strip().casefold() for value in reviewers if str(value).strip()}) < 2:
        raise ValueError(f"Two independent reviewer IDs required for {pair.get('row_id')}")
    if pair.get("language") not in {"en", "ja"}:
        raise ValueError(f"Unsupported or missing language for {pair.get('row_id')}")
if len({(pair["language"], pair["seed_id"]) for pair in pairs}) != len(pairs):
    raise ValueError("Evaluation examples must be unique within each language and source seed.")
for language in ("en", "ja"):
    count = len({pair["seed_id"] for pair in pairs if pair["language"] == language})
    if count < 8:
        raise ValueError(f"Only {count} fresh {language} evaluation pairs; require at least 8 per language.")
if len(pairs) > 32:
    raise ValueError(f"{len(pairs)} rows exceed the predeclared 32-seed causal pilot cap; define a new sampling plan rather than silently truncating.")
discovery_meta = json.loads(DISCOVERY_META.read_text())
if set(discovery_meta["analysis_seed_ids"]) & {pair["seed_id"] for pair in pairs}:
    raise ValueError("Fresh evaluation seeds overlap the circuit-discovery set.")
phase1_report = json.loads(PHASE1.read_text())
metric_seeds = {item["seed_id"] for item in phase1_report.get("pairs", [])}
if metric_seeds & {pair["seed_id"] for pair in pairs}:
    raise ValueError("Fresh evaluation seeds overlap notebook 01 analysis/tune metric-validation seeds.")
if not PRIOR_SEED_REGISTRY.is_file():
    raise FileNotFoundError("Complete the historical training/tuning seed inventory at " + str(PRIOR_SEED_REGISTRY))
from eval.track2_extension.seed_audit import validate_fresh_eval_seed_audit
prior_seed_audit = validate_fresh_eval_seed_audit(
    pairs, discovery_seed_ids=_discovery_meta["analysis_seed_ids"],
    phase1_seed_ids=metric_seeds, prior_registry=json.loads(PRIOR_SEED_REGISTRY.read_text()),
    minimum_per_language=8, maximum_rows=32,
)
print("Fresh evaluation seed audit passed:", prior_seed_audit)

from circuitkit.tasks.registry import _TASKS
from circuitkit.evaluation.pillars import Pillar1_CausalPatching, Pillar2_Ablation
from eval.track2_extension.intervention_summary import valid_raw_ratio_values, invalid_raw_ratio_count

# CircuitKIT's public report clips its headline pillar scores into [0, 1]. Preserve
# the complete per-pillar outputs too: raw metric values, signed ratios, and
# denominator validity flags are essential for a signed refusal-minus-compliance
# metric. This mirrors the capture method used in official safety case study 23.
PILLAR_CAPTURE = {}
_original_p1_run = Pillar1_CausalPatching.run
_original_p2_run = Pillar2_Ablation.run

def _capture_p1(*args, **kwargs):
    result = _original_p1_run(*args, **kwargs)
    PILLAR_CAPTURE["patching"] = result
    return result

def _capture_p2(*args, **kwargs):
    result = _original_p2_run(*args, **kwargs)
    PILLAR_CAPTURE["ablation"] = result
    return result

Pillar1_CausalPatching.run = staticmethod(_capture_p1)
Pillar2_Ablation.run = staticmethod(_capture_p2)

# Run one fresh evaluation seed at a time. Each language is evaluated against
# the circuit selected from its own language-specific analysis cohort.
model = ck.load_model(MODEL, revision=REVISION, dtype="float32", device="cuda", token=True, algorithm="eap-ig").eval()
model.cfg.ungroup_grouped_query_attention = True
torch.cuda.reset_peak_memory_stats()
base_config = {
    "model": {"name": MODEL, "precision": "float32", "device": "cuda"},
    "discovery": {
        "algorithm": "eap-ig", "level": "node", "method": "EAP-IG-inputs",
        "batch_size": 2, "ig_steps": 3, "verbose": True,
        "max_seq_len": 2048, "data_params": {"num_examples": 2},
    },
    "pruning": {"target_sparsity": 0.10, "scope": "both", "intervention": "patching"},
    "output_path": str(ARTIFACTS["en"]),
    "eval": {
        "full_faithfulness_eval": True, "num_examples": 2,
        "pillars": ["patching", "ablation", "baselines"],
        "baseline_types": ["random"], "n_random_draws": 10,
    },
}

def json_safe(value):
    if value is None or isinstance(value, (str, bool, int)):
        return value
    if isinstance(value, float):
        import math
        return value if math.isfinite(value) else None
    if hasattr(value, "item"):
        try:
            return json_safe(value.item())
        except Exception:
            pass
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if hasattr(value, "__dict__"):
        return json_safe(value.__dict__)
    return repr(value)

per_seed = []
for index, pair in enumerate(sorted(pairs, key=lambda row: (row["language"], row["seed_id"]))):
    language = pair["language"]
    artifact = ARTIFACTS[language]
    task_name = f"{TASK}_{language}_{index:03d}"
    _TASKS.pop(task_name, None)
    register_task(RefusalComplianceTaskSpec(pairs=[pair], name=task_name))
    config = json.loads(json.dumps(base_config))
    config["output_path"] = str(artifact)
    config["discovery"]["task"] = task_name
    config["discovery"]["seed"] = 20261113 + index
    config["discovery"]["data_params"]["seed"] = 20261113 + index
    config["eval"]["seed"] = 20261113 + index
    PILLAR_CAPTURE.clear()
    report = ck.evaluate_circuit(
        config, pruned_artifact_path=str(artifact),
        scores_path=str(artifact.with_name(artifact.stem + "_scores.pt")), _model=model,
    )
    patching = PILLAR_CAPTURE.get("patching", {})
    ablation = PILLAR_CAPTURE.get("ablation", {})
    per_seed.append({
        "seed_id": pair["seed_id"], "row_id": pair["row_id"], "language": language,
        "circuit_artifact": artifact.name,
        # Primary outcomes are the raw signed refusal-minus-compliance metric.
        "patching_metric_raw_score": json_safe(patching.get("raw_score")),
        "patching_clean_metric": json_safe(patching.get("clean_score")),
        "patching_corrupt_metric": json_safe(patching.get("corrupt_score")),
        "patching_raw_ratio": json_safe(patching.get("raw_ratio")),
        "patching_degenerate_denominator": json_safe(patching.get("degenerate_denominator")),
        "patching_inverted_denominator": json_safe(patching.get("inverted_denominator")),
        "patching_status": json_safe(patching.get("status", "ok")),
        "ablation_metric_raw_score": json_safe(ablation.get("raw_score")),
        "ablation_clean_metric": json_safe(ablation.get("clean_score")),
        "ablation_corrupt_metric": json_safe(ablation.get("corrupt_score")),
        "ablation_raw_ratio": json_safe(ablation.get("raw_ratio")),
        "ablation_degenerate_denominator": json_safe(ablation.get("degenerate_denominator")),
        "ablation_inverted_denominator": json_safe(ablation.get("inverted_denominator")),
        "ablation_status": json_safe(ablation.get("status", "ok")),
        # CircuitKIT's normalized pillar scores are clipped and are secondary.
        "patching_score_clipped": json_safe(getattr(report, "patching_score", None)),
        "ablation_score_clipped": json_safe(getattr(report, "ablation_score", None)),
        "baseline_comparison": json_safe(getattr(report, "baseline_comparison", None)),
        "metadata": json_safe(getattr(report, "metadata", {})),
        "report": json_safe(report),
    })
    print(language, pair["seed_id"], per_seed[-1]["patching_metric_raw_score"], per_seed[-1]["ablation_metric_raw_score"])

def bootstrap_mean_ci(values, seed=20261113, draws=10000):
    import numpy as np
    vals = np.asarray([value for value in values if isinstance(value, (int, float)) and np.isfinite(value)], dtype=float)
    if len(vals) < 2:
        return {"n": int(len(vals)), "mean": float(vals.mean()) if len(vals) else None, "ci95": None}
    rng = np.random.default_rng(seed)
    means = np.empty(draws)
    for draw in range(draws):
        means[draw] = rng.choice(vals, size=len(vals), replace=True).mean()
    return {"n": int(len(vals)), "mean": float(vals.mean()),
            "ci95": [float(value) for value in np.quantile(means, [0.025, 0.975])],
            "method": "source-seed bootstrap of per-seed CircuitKIT reports"}

summary_by_language = {}
for language in ("en", "ja"):
    language_rows = [row for row in per_seed if row["language"] == language]
    summary_by_language[language] = {
        "n_seeds": len(language_rows),
        "patching_metric_raw_score": bootstrap_mean_ci([row["patching_metric_raw_score"] for row in language_rows], seed=20261113),
        "patching_clean_metric": bootstrap_mean_ci([row["patching_clean_metric"] for row in language_rows], seed=20261115),
        "patching_corrupt_metric": bootstrap_mean_ci([row["patching_corrupt_metric"] for row in language_rows], seed=20261116),
        "patching_raw_ratio_valid_only": bootstrap_mean_ci(
            valid_raw_ratio_values(language_rows, "patching"), seed=20261117),
        "patching_ratio_invalid_n": invalid_raw_ratio_count(language_rows, "patching"),
        "ablation_metric_raw_score": bootstrap_mean_ci([row["ablation_metric_raw_score"] for row in language_rows], seed=20261114),
        "ablation_clean_metric": bootstrap_mean_ci([row["ablation_clean_metric"] for row in language_rows], seed=20261118),
        "ablation_corrupt_metric": bootstrap_mean_ci([row["ablation_corrupt_metric"] for row in language_rows], seed=20261119),
        "ablation_raw_ratio_valid_only": bootstrap_mean_ci(
            valid_raw_ratio_values(language_rows, "ablation"), seed=20261120),
        "ablation_ratio_invalid_n": invalid_raw_ratio_count(language_rows, "ablation"),
        "clipped_scores_secondary": {
            "patching": bootstrap_mean_ci([row["patching_score_clipped"] for row in language_rows], seed=20261121),
            "ablation": bootstrap_mean_ci([row["ablation_score_clipped"] for row in language_rows], seed=20261122),
        },
        "random_baseline_reports": [row["baseline_comparison"] for row in language_rows],
    }
result = {
    "model": MODEL, "revision": REVISION, "circuitkit_source_commit": source_commit,
    "task": TASK, "n_pairs": len(pairs),
    "seed_ids_sha256": __import__("hashlib").sha256("\n".join(sorted({pair["seed_id"] for pair in pairs})).encode()).hexdigest(),
    "evaluation_export_sha256": __import__("hashlib").sha256(raw).hexdigest(),
    "pillars": ["patching", "ablation", "random baseline"],
    "per_seed": per_seed, "seed_clustered_summary_by_language": summary_by_language,
    "versions": versions, "gpu": torch.cuda.get_device_name(0),
    "peak_allocated_bytes": torch.cuda.max_memory_allocated(),
    "peak_reserved_bytes": torch.cuda.max_memory_reserved(),
    "limitation": "Causal effect on the validated likelihood metric only; does not establish reduced unsafe compliance in a broad population.",
}
(RUN / "causal_validation.json").write_text(json.dumps(result, indent=2, allow_nan=False) + "\n")
print(json.dumps(result["seed_clustered_summary_by_language"], indent=2))
